# AutoScience — cloud benchmark runner

Runs the `full` benchmark profile (complete datasets incl. 11M-row HIGGS, 5 seeds, full HPO budgets) on Colab or any Linux GPU box, then packages the MLflow store for download.

**Runtime:** GPU recommended (torch MLP); everything else is CPU-parallel. The sweep is resumable — if the session dies, rerun the notebook and finished runs are skipped.

In [ ]:
REPO_URL = "https://github.com/<your-user>/AutoScience.git"  # <- set me
BENCHMARK_CONFIG = "experiments/benchmark_full.yaml"

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!git clone {REPO_URL} autoscience_repo || (cd autoscience_repo && git pull)
%cd autoscience_repo
!uv sync --all-groups

In [ ]:
# Swap the locked CPU torch wheel for a CUDA build when a GPU is present.
import subprocess
gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
if gpu:
    !uv pip install torch --torch-backend=auto
!uv run python -c "import torch; print('cuda available:', torch.cuda.is_available())"

In [ ]:
# Pre-download and validate every dataset (HIGGS is a 2.6 GB download).
!uv run autoscience data validate --all --include-large

In [ ]:
# The sweep. Resumable: rerun this cell after any interruption.
!uv run autoscience benchmark --config {BENCHMARK_CONFIG}

In [ ]:
# Package results: the SQLite tracking DB + artifacts, ready to merge locally
# (drop mlflow.db + mlartifacts/ into the repo root, or point
# MLFLOW_TRACKING_URI at the db) and run `autoscience report`.
!zip -qr autoscience_results.zip mlflow.db mlartifacts reports || true
print("Download autoscience_results.zip from the file browser, or mount Drive:")
# from google.colab import drive; drive.mount('/content/drive')
# !cp autoscience_results.zip /content/drive/MyDrive/